# Multiaxial Universal ML plasticity model

To generalise the formulation in [Metal1D](https://github.com/sinaplatform/plasticity-models/blob/main/Metal1D.ipynb) into a multiaxial case, first consider the elastic case as follow

$$ d\sigma= \mathbb{C}^e d\varepsilon^e$$

where in Matrix Form (Voigt Notation) $\mathbb{C}^e$ is 6 by 6 full matrix, it means that the stress increment in one direction is coupled with all loading directions.

$\textbf{IMPORTANT obeservation:}$ Accourding to the 1D formulation in [Metal1D](https://github.com/sinaplatform/plasticity-models/blob/main/Metal1D.ipynb), it seems that $\mathbb{C}$ itself if coupled with intenal state variables is able to handle history dependency!


Let us forget about the seperation of elastic and plastic part of material response and assume that the material is nonlinear under any loading magnitude. 

$$ d\sigma= \mathbb{C} d\varepsilon$$

Note that $\sigma$ and $\varepsilon$ are second order tensors in 3D and the material stiffness (tangental or Jacobian) $\mathbb{C}$ is a fourth-order tensor

$$
d \sigma_{ij} =  \mathbb{C}_{ijkl} d \varepsilon_{kl}
$$

where 

$$ 
\mathbb{C}_{ijkl} = \frac{\partial d \sigma_{ij}}{\partial d \varepsilon_{kl}}.
$$

In Matrix Form (Voigt Notation):

$$
d \sigma = 
\begin{bmatrix}
d \sigma_{11} \\
d \sigma_{22} \\
d \sigma_{33} \\
d \sigma_{23} \\
d \sigma_{31} \\
d \sigma_{12}
\end{bmatrix}
,~~~
d \varepsilon = 
\begin{bmatrix}
d \varepsilon_{11} \\
d \varepsilon_{22} \\
d \varepsilon_{33} \\
2 d \varepsilon_{23} \\
2 d \varepsilon_{31} \\
2 d \varepsilon_{12}
\end{bmatrix}
$$

These are different in ABAUQS.

$$
\mathbb{C} =
\begin{bmatrix}
C_{11} & C_{12} & C_{13} & C_{14} & C_{15} & C_{16} \\
C_{21} & C_{22} & C_{23} & C_{24} & C_{25} & C_{26} \\
C_{31} & C_{32} & C_{33} & C_{34} & C_{35} & C_{36} \\
C_{41} & C_{42} & C_{43} & C_{44} & C_{45} & C_{46} \\
C_{51} & C_{52} & C_{53} & C_{54} & C_{55} & C_{56} \\
C_{61} & C_{62} & C_{63} & C_{64} & C_{65} & C_{66} \\
\end{bmatrix}.
$$


<!-- Therefore a universal dynamic ML material plasticity model will look like this:

$$ \left\{
    \begin{aligned}
        d\sigma &= f(h) d\varepsilon\\
        dh &=g(h) d\varepsilon \\
    \end{aligned}
    \right. 
$$

or 

$$ \left\{
    \begin{aligned}
        \frac{d\sigma}{d\varepsilon}  &= f(h) \\
        \frac{dh}{d\varepsilon} &=g(h) \\
    \end{aligned}
    \right. 
$$

where $f$ and $g$ are neural operators (NO) with dimensions $f: \mathbb{R}^k \rightarrow \mathbb{R}^6 $ and $g: \mathbb{R}^k \rightarrow \mathbb{R}^6 $ for a 3D case. Here, $k$ is the number of states and we consider the Jacobian of states as below which for training purposes flattens.

$$
\frac{d\boldsymbol{h}}{d\boldsymbol{\varepsilon}} =
\begin{bmatrix}
\frac{\partial h^1}{\partial \varepsilon^1} & \frac{\partial h^1}{\partial \varepsilon^2} & \cdots \\
\frac{\partial h^2}{\partial \varepsilon^1} & \frac{\partial h^2}{\partial \varepsilon^2} & \cdots \\
\vdots & \vdots & \ddots
\end{bmatrix}
$$

Yet, we can use only one NN for all state evolutions or one NN for each state variable in all directions.  -->

## New state space formulation
Considering the [combined isotropic/kinematic modeling in 1D](https://github.com/sinaplatform/plasticity-models/blob/main/VonMisesMetal1D.ipynb)
If $ \boldsymbol{y} = \begin{bmatrix}
\boldsymbol{\sigma} \\
\boldsymbol{h}
\end{bmatrix} $ the universal state space ML model will be as follows:

$$ \left\{
    \begin{aligned}
        d\boldsymbol{y}(s) &= \mathscr{NO}(\boldsymbol{y}(s), d \boldsymbol{\varepsilon}(s), \theta_{\boldsymbol{w},\boldsymbol{b}}) ~ d [\boldsymbol{\varepsilon}(s),t(s)] \\
        \boldsymbol{y}(0) & = \begin{bmatrix}
                    \boldsymbol{\sigma} (0) \\
                    \boldsymbol{h} (0)
                    \end{bmatrix} \\
        \hat{\boldsymbol{\sigma}} &= \boldsymbol{\sigma}(1)          
    \end{aligned}
    \right. \quad s \in[0,1]
$$


where $\mathscr{NO}$ is neural operators with dimensions $\mathscr{NO}: \mathbb{R}^{2o+k+1} \rightarrow \mathbb{R}^{(o(o+1)/2)+o+k \times (o+1)} $. 
Here, $s$ is the increment step, and  $k$ is the number of states and $o$ is the number of obervable states which correspond to the stress components.
Note that the first $o$ raws in the output is a symmetric matrix and therefore the learnable output dimension is only $(o(o+1)/2)+o+k \times (o+1)$.


$$ 
\min_{\theta = \{w, b\}} \text{MSE} = \frac{1}{q \cdot m} \sum_{p=1}^{q} \sum_{n=1}^{m} \left(\sigma_p^n - \hat{\sigma}_p^n\right)^2
$$ 

This architecture directly learns material jacobian and jacobian of internal state variables ($h^k$).

$$
\frac{d\boldsymbol{y}}{d[\boldsymbol{\varepsilon},t]} =
\begin{bmatrix}
\frac{\partial \sigma^1}{\partial \varepsilon^1} & \frac{\partial \sigma^1}{\partial \varepsilon^2} & \frac{\partial \sigma^1}{\partial \varepsilon^3} &
\frac{\partial \sigma^1}{\partial \varepsilon^4} &  \frac{\partial \sigma^1}{\partial \varepsilon^5} & \frac{\partial \sigma^1}{\partial \varepsilon^6} &
\frac{\partial \sigma^1}{\partial t} 
 \\
        & \frac{\partial \sigma^2}{\partial \varepsilon^2} & \frac{\partial \sigma^2}{\partial \varepsilon^3} &
\frac{\partial \sigma^2}{\partial \varepsilon^4} &  \frac{\partial \sigma^2}{\partial \varepsilon^5} & \frac{\partial \sigma^2}{\partial \varepsilon^6} &
\frac{\partial \sigma^2}{\partial t}
 \\
     &   & \frac{\partial \sigma^3}{\partial \varepsilon^3} &
\frac{\partial \sigma^3}{\partial \varepsilon^4} &  \frac{\partial \sigma^3}{\partial \varepsilon^5} & \frac{\partial \sigma^3}{\partial \varepsilon^6} & 
\frac{\partial \sigma^3}{\partial t}
 \\
     &   &   &
\frac{\partial \sigma^4}{\partial \varepsilon^4} &  \frac{\partial \sigma^4}{\partial \varepsilon^5} & \frac{\partial \sigma^4}{\partial \varepsilon^6} &
\frac{\partial \sigma^4}{\partial t}
 \\
     & \text{sym}  &   &
     &  \frac{\partial \sigma^5}{\partial \varepsilon^5} & \frac{\partial \sigma^5}{\partial \varepsilon^6} &
\frac{\partial \sigma^5}{\partial t}
 \\
     &   &   &
     &       & \frac{\partial \sigma^6}{\partial \varepsilon^6} &
\frac{\partial \sigma^6}{\partial t}
 \\ \hdashline

\frac{\partial h^1}{\partial \varepsilon^1} & \frac{\partial h^1}{\partial \varepsilon^2} & \frac{\partial h^1}{\partial \varepsilon^3} &
\frac{\partial h^1}{\partial \varepsilon^4} &  \frac{\partial h^1}{\partial \varepsilon^5} & \frac{\partial h^1}{\partial \varepsilon^6} &
\frac{\partial h^1}{\partial t}
 \\
\frac{\partial h^2}{\partial \varepsilon^1} & \frac{\partial h^2}{\partial \varepsilon^2} & \frac{\partial h^1}{\partial \varepsilon^3} &
\frac{\partial h^2}{\partial \varepsilon^4} &  \frac{\partial h^2}{\partial \varepsilon^5} & \frac{\partial h^2}{\partial \varepsilon^6} &
\frac{\partial h^2}{\partial t}  \\
\vdots & \vdots & \vdots &\vdots &\vdots &\vdots &\vdots 
\end{bmatrix}
$$


In [ ]:
import argparse
import time
import numpy as np
from scipy.interpolate import interp1d
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import pytorch_lightning as pl

from sklearn.preprocessing import MinMaxScaler, StandardScaler, MaxAbsScaler

import math
import time

import diffrax
import equinox as eqx  # https://github.com/patrick-kidger/equinox
import jax
jax.config.update("jax_platform_name", "cpu")  # Ensure running on CPU
jax.config.update("jax_enable_x64", True)
from jax import device_put
import jax.nn as jnn
import jax.numpy as jnp
import jax.random as jr
import jax.scipy as jsp
import matplotlib
import matplotlib.pyplot as plt
import optax  # https://github.com/deepmind/optax
from typing import Tuple, NamedTuple

import matplotlib.pyplot as plt
from ipywidgets import interact
%matplotlib qt
import pyqtgraph as pg
from pyqtgraph.Qt import QtGui
plt.rcdefaults()
# figure styling
font=16
font_axis=20
plt.rcParams.update({'font.size': font})  # Set the desired font size
plt.rcParams['figure.facecolor'] = 'white'  # Background color for figures
plt.rcParams['axes.facecolor'] = 'white'    # Background color for axes

In [ ]:
class StateSpaceFunc(eqx.Module):
    mlp: eqx.nn.MLP
    stress_dim: int  # number of observable states
    k_inv: int       # number of hidden states
    input_size: int 
    output_size: int
    width_size: int 
    num_layers: int
    
    def __init__(self, stress_dim, k_inv, width_size, num_layers, *, key, **kwargs):
        super().__init__(**kwargs)
        self.stress_dim = stress_dim  # Number of observable states
        self.k_inv = k_inv            # Number of hidden states
        self.input_size = 2 * stress_dim + k_inv
        self.output_size = (((stress_dim + 1) * stress_dim)  // 2) + (k_inv * stress_dim) # Size for symmetric matrix + hidden state interactions
        self.width_size = width_size
        self.num_layers = num_layers

        self.mlp = eqx.nn.MLP(
            in_size = self.input_size,
            out_size = self.output_size,
            width_size = self.width_size,
            depth = self.num_layers,
            activation = jnn.selu,
            # Note the use of a tanh final activation function. This is important to
            # stop the model blowing up. (Just like how GRUs and LSTMs constrain the
            # rate of change of their hidden states.)
            # final_activation = jnn.tanh,
            key = key,
        )

    
    def construct_jacobian(self, output):
        """Constructs the structured Jacobian matrix from neural network output"""
        stress_dim, k_inv = self.stress_dim, self.k_inv
        
        # Split the output into symmetric part and hidden state interactions
        sym_size = (stress_dim + 1) * stress_dim // 2
        sym_part = output[:sym_size]
        hidden_part = output[sym_size:]
        
        # Construct symmetric matrix for observable states
        sym_matrix = jnp.zeros((stress_dim, stress_dim))
        idx = 0
        for i in range(stress_dim):
            for j in range(i, stress_dim):
                sym_matrix = sym_matrix.at[i, j].set(sym_part[idx])
                sym_matrix = sym_matrix.at[j, i].set(sym_part[idx])
                idx += 1
    
        # Construct full Jacobian
        jac_matrix = jnp.zeros((stress_dim + k_inv, stress_dim))
        # Set symmetric part
        jac_matrix = jac_matrix.at[:stress_dim, :stress_dim].set(sym_matrix)
        # Set hidden state interactions
        hidden_matrix = hidden_part.reshape(k_inv, stress_dim)
        jac_matrix = jac_matrix.at[stress_dim:, :stress_dim].set(hidden_matrix)
        
        return jac_matrix
    
    def __call__(self, t, y, args):
        # Split y into observable and hidden states
        # sigma = y[1 : self.stress_dim + 1]
        # h = y[self.stress_dim + 1 :]

        # Get network output and construct Jacobian y(o) is the time
        interp = args["control"]
        strain=interp.evaluate(t)
        dstrain=interp.derivative(t)
  
        # Get network output and construct Jacobian y(o) is the time
        input = jnp.concatenate([y[1:],dstrain[1:]]) #jnp.sign

        output = self.mlp(input)
        jacobian = self.construct_jacobian(output)
        
        m, n = jacobian.shape
        # Create a new matrix with dimensions (m+1, n+1)
        f = jnp.zeros((m + 1, n + 1))
        # Set the top-left corner to 1
        f = f.at[0,0].set(1)
        # Place the original matrix F in the bottom right corner
        f = f.at[1:, 1:].set(jacobian)
       
        return f

class StateSpaceNeuralCDE(eqx.Module):
    # initial: eqx.nn.MLP
    func: StateSpaceFunc
    # linear: eqx.nn.Linear

    def __init__(self, stress_dim, k_inv, width_size, num_layers, *, key, **kwargs):
        super().__init__(**kwargs)
        ikey, fkey, lkey = jr.split(key, 3)
        
        self.func = StateSpaceFunc(stress_dim, k_inv, width_size, num_layers, key=fkey)
    
    def __call__(self, ts, y0, coeffs):
        # Each sample of data consists of some timestamps `ts`, and some `coeffs`
        # parameterising a control path. These are used to produce a continuous-time
        # input path `control`.
        
        control = diffrax.CubicInterpolation(ts, coeffs)
        strain = {"control": control}
        term = diffrax.ControlTerm(self.func, control).to_ode()
        
        solver = diffrax.Dopri5() # Euler(), 
        # dt0 = None
        # y0 = self.initial(control.evaluate(ts[0]))
        saveat = diffrax.SaveAt(ts=ts)

        solution = diffrax.diffeqsolve(
            term,
            solver,
            t0 = ts[0],
            t1 = ts[-1],
            dt0 = ts[1] - ts[0],
            y0 = y0,
            args=strain,
            stepsize_controller = diffrax.PIDController(rtol=1e-6, atol=1e-6),
            saveat = saveat,
            # adjoint=diffrax.BacksolveAdjoint(),
        )
        return solution.ys

def dataloader(arrays, batch_size, *, key):
    dataset_size = arrays[0].shape[0]
    assert all(array.shape[0] == dataset_size for array in arrays)
    indices = jnp.arange(dataset_size)
    while True:
        perm = jr.permutation(key, indices)
        (key,) = jr.split(key, 1)
        start = 0
        end = batch_size
        while end < dataset_size:
            batch_perm = perm[start:end]
            yield tuple(array[batch_perm] for array in arrays)
            start = end
            end = start + batch_size
            
def rmse_loss(y_pred, y_true):
    """Compute MSE loss focusing on observable states"""
    MSE = jnp.mean((y_pred - y_true) ** 2)
    return MSE

@eqx.filter_value_and_grad
def grad_loss(model, ti, yi, coeff_i, stress_dim, k_inv):
    y0 = yi[:, 0]
    zeros = jnp.zeros((y0.shape[0], k_inv))              # Create an array of zeros with shape e.g., (1000,1,3)
    y0_expanded = jnp.concatenate([y0, zeros], axis=-1)  # Concatenate along the last dimension
    y_pred = jax.vmap(model, in_axes=(None, 0, 0))(ti[0], y0_expanded, coeff_i)
    
    return rmse_loss(y_pred[:, :, 1:stress_dim+1], yi[:, :, 1:stress_dim+1]) # 1:stress_dim+1

@eqx.filter_jit
def make_step(model, data_i,  opt_state, optim, stress_dim, k_inv):
    ti, yi, *coeff_i = data_i
    loss, grads = grad_loss(model, ti, yi, coeff_i, stress_dim, k_inv)
    updates, opt_state = optim.update(grads, opt_state)
    model = eqx.apply_updates(model, updates)
    return loss, model, opt_state

def train_model(loader_key, print_every, model, ts, ys, coeffs, stress_dim, k_inv, batch_size, *, learning_rate=1e-3, steps=1000):
                
    optim = optax.adam(learning_rate)
    opt_state = optim.init(eqx.filter(model, eqx.is_inexact_array))
    
    train_losses = []
    for step, data_i in zip(
        range(steps), dataloader((ts, ys) + coeffs, batch_size, key=loader_key)
    ):
        start = time.time()
        loss, model, opt_state = make_step(model, data_i, opt_state, optim, stress_dim, k_inv)
        end = time.time()
            
        if (step % print_every) == 0 or step == steps - 1:
            print(f"Step: {step}, Loss: {loss}, Computation time: {end - start}")
        train_losses.append(loss)
    
    return model, train_losses

In [ ]:
def main(
    dataset_size = 1100,
    batch_size = 128,
    lr_strategy = (3e-3, 3e-3),
    steps_strategy = (500, 500),
    length_strategy = (0.1, 1),
    stress_dim = 6, # observable states
    k_inv = 3,  # number of internal state variables
    width_size = 200,
    num_layers = 4,
    seed=5678,
    plot=True,
    print_every=50,
):
    
    # import and navigate through the dataset
    npy_time = np.load('time.npy')
    # time=time[1,:,:]
    output = np.load('ave_tensor.npy')

    # Initialize arrays for the new strain and stress tensors with 9 components
    strain = np.zeros((output.shape[0], output.shape[1], 6))
    stress = np.zeros((output.shape[0], output.shape[1], 6))

    # Fill the strain tensor components
    strain[:, :, 0] = output[:, :, 0]  # E11
    strain[:, :, 1] = output[:, :, 1]  # E22
    strain[:, :, 2] = output[:, :, 2]  # E33
    strain[:, :, 3] = output[:, :, 3]  # E12 
    strain[:, :, 4] = output[:, :, 4]  # E13
    strain[:, :, 5] = output[:, :, 5]  # E23

    # Fill the stress tensor components
    stress[:, :, 0] = output[:, :, 6] * 1e6   # S11
    stress[:, :, 1] = output[:, :, 7] * 1e6  # S22
    stress[:, :, 2] = output[:, :, 8] * 1e6  # S33
    stress[:, :, 3] = output[:, :, 9] * 1e6  # S12 
    stress[:, :, 4] = output[:, :, 10] * 1e6  # S13
    stress[:, :, 5] = output[:, :, 11] * 1e6 # S23

    # Importing and Scaling data
    # npy_time = np.expand_dims(np.load('t.npy'), axis=-1)
    # strain = np.expand_dims(np.load('strain.npy'), axis=-1)
    # stress = np.expand_dims(np.load('S.npy'), axis=-1)
    # state = np.expand_dims(np.load('state.npy'), axis=-1) # z in the paper
    # full_state = np.concatenate([stress, state], axis=-1)

    
    # max_abs_strain = np.abs(strain[:,:,:6]).max()
    # strain_train_scaled = strain[:,:,:6] / max_abs_strain

    # max_abs_stress = np.abs(stress[:,:,:6]).max()
    # stress_train_scaled = stress[:,:,:6] / max_abs_stress

    # Scaling data
    scaler_E = MaxAbsScaler()
    scaler_S = MaxAbsScaler()
    # scaler_fS = MaxAbsScaler()

    strain_train_reshaped = strain.reshape(-1, strain.shape[2]) # Reshape to 2D for scaling
    stress_train_reshaped = stress.reshape(-1, stress.shape[2])
    # full_state_train_reshaped = full_state.reshape(-1, full_state.shape[2])
    num_datasets=strain.shape[0]
    timesteps=strain.shape[1]
    strain_train_scaled = scaler_E.fit_transform(strain_train_reshaped).reshape(num_datasets, timesteps, strain.shape[-1])
    stress_train_scaled = scaler_S.fit_transform(stress_train_reshaped).reshape(num_datasets, timesteps, stress.shape[-1])
    # full_state_train_scaled = scaler_fS.fit_transform(full_state_train_reshaped).reshape(num_datasets, timesteps, full_state.shape[-1])

    jax_time = device_put(npy_time)
    jax_strain = device_put(strain_train_scaled)
    jax_stress = device_put(stress_train_scaled)
    # jax_state = device_put(full_state_train_scaled)

    x_strain = jnp.concatenate([jax_time, jax_strain], axis=-1)  # time is a channel
    coeffs = jax.vmap(diffrax.backward_hermite_coefficients)(jax_time.squeeze(-1), x_strain)

    ts = jax_time.squeeze(-1)
    # ys = jnp.concatenate([jax_time, jax_state], axis=-1)
    ys = jnp.concatenate([jax_time, jax_stress], axis=-1)

    # Defining model 
    key = jr.PRNGKey(seed)
    train_data_key, test_data_key, model_key, loader_key = jr.split(key, 4)
    
    model = StateSpaceNeuralCDE (stress_dim, k_inv, width_size, num_layers, key = model_key)

    # Training loop
    model, train_losses = train_model(loader_key, print_every, model, ts, ys, coeffs, stress_dim, k_inv, batch_size, learning_rate=1e-3, steps=2000)

    return model, train_losses

model, train_losses = main()

# Save the model to a file.
eqx.tree_serialise_leaves("model.eqx", model)
# Save the results to a file
jnp.save('training_results.npy', train_losses)

In [ ]:
# # Reconstruct a model with the same architecture, e.g., using a dummy key.
model_dummy = StateSpaceNeuralCDE (6, 6, 200, 4, key=jax.random.PRNGKey(0))
model = eqx.tree_deserialise_leaves("model_Chaboce.eqx", model_dummy)
result = np.load('training_results_Chaboce.npy',  allow_pickle=True)
train_losses = result

In [ ]:
plt.figure(figsize=(5, 4))
plt.plot(train_losses , label='Training Loss', color='blue')
plt.yscale('log')
plt.xlabel('Epoch', fontsize=14, fontname='Times New Roman')
plt.ylabel('MSE Loss', fontsize=14, fontname='Times New Roman')
plt.tick_params(axis='both', which='major', direction='in')
plt.tick_params(axis='both', which='minor', direction='in')
# plt.autoscale(tight=True)
plt.legend(prop={'family': 'Times New Roman', 'size': 14}, frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# import and navigate through the dataset
npy_time = np.load('time.npy')
# time=time[1,:,:]
output = np.load('ave_tensor.npy')

# Initialize arrays for the new strain and stress tensors with 9 components
strain = np.zeros((output.shape[0], output.shape[1], 6))
stress = np.zeros((output.shape[0], output.shape[1], 6))

# Fill the strain tensor components
strain[:, :, 0] = output[:, :, 0]  # E11
strain[:, :, 1] = output[:, :, 1]  # E22
strain[:, :, 2] = output[:, :, 2]  # E33
strain[:, :, 3] = output[:, :, 3]  # E12 
strain[:, :, 4] = output[:, :, 4]  # E13
strain[:, :, 5] = output[:, :, 5]  # E23

# Fill the stress tensor components
stress[:, :, 0] = output[:, :, 6] * 1e6   # S11
stress[:, :, 1] = output[:, :, 7] * 1e6  # S22
stress[:, :, 2] = output[:, :, 8] * 1e6  # S33
stress[:, :, 3] = output[:, :, 9] * 1e6  # S12 
stress[:, :, 4] = output[:, :, 10] * 1e6  # S13
stress[:, :, 5] = output[:, :, 11] * 1e6 # S23

# Importing and Scaling data
# npy_time = np.expand_dims(np.load('t.npy'), axis=-1)
# strain = np.expand_dims(np.load('strain.npy'), axis=-1)
# stress = np.expand_dims(np.load('S.npy'), axis=-1)
# state = np.expand_dims(np.load('state.npy'), axis=-1) # z in the paper
# full_state = np.concatenate([stress, state], axis=-1)

# max_abs_strain = np.abs(strain[:,:,:6]).max()
# strain_train_scaled = strain[:,:,:6] / max_abs_strain

# max_abs_stress = np.abs(stress[:,:,:6]).max()
# stress_train_scaled = stress[:,:,:6] / max_abs_stress

# Scaling data
scaler_E = MaxAbsScaler()
scaler_S = MaxAbsScaler()
# scaler_fS = MaxAbsScaler()

strain_train_reshaped = strain.reshape(-1, strain.shape[2]) # Reshape to 2D for scaling
stress_train_reshaped = stress.reshape(-1, stress.shape[2])
# full_state_train_reshaped = full_state.reshape(-1, full_state.shape[2])
num_datasets=strain.shape[0]
timesteps=strain.shape[1]
strain_train_scaled = scaler_E.fit_transform(strain_train_reshaped).reshape(num_datasets, timesteps, strain.shape[-1])
stress_train_scaled = scaler_S.fit_transform(stress_train_reshaped).reshape(num_datasets, timesteps, stress.shape[-1])
# full_state_train_scaled = scaler_fS.fit_transform(full_state_train_reshaped).reshape(num_datasets, timesteps, full_state.shape[-1])

jax_time = device_put(npy_time)
jax_strain = device_put(strain_train_scaled)
jax_stress = device_put(stress_train_scaled)
# jax_state = device_put(full_state_train_scaled)

x_strain = jnp.concatenate([jax_time, jax_strain], axis=-1)  # time is a channel
coeffs = jax.vmap(diffrax.backward_hermite_coefficients)(jax_time.squeeze(-1), x_strain)

ts = jax_time.squeeze(-1)
ys = jnp.concatenate([jax_time, jax_stress], axis=-1)

traj_ID = 52 # Trajectory ID to visualize
mag_scale=1e6
k_inv = 6

_coeffs = tuple(arr[traj_ID, :, :] for arr in coeffs)
y0 = ys[traj_ID, 0]

zeros = jnp.zeros( k_inv)  # Create an array of zeros with shape (1000,1,3)
y0_expanded = jnp.concatenate([y0, zeros], axis=-1)  # Concatenate along the last dimension
model_y = model(ts[traj_ID], y0_expanded, _coeffs)

# Define component names corresponding to the 6 components
components = ['11', '22', '33', '12', '13', '23']

# Create a 2x3 grid of subplots to reflect tensor symmetry
fig, axs = plt.subplots(2, 3, figsize=(12, 7))

for i, ax in enumerate(axs.flat):
    if i < 6:
        component = components[i]

        ax.plot(strain_train_scaled[traj_ID, :, i], 'b.', alpha=0.2, label="Strain")
        ax.plot(stress_train_scaled[traj_ID, :, i], 'k.', label='reference data')
        ax.plot(model_y[:, i+1], 'r', label='model estimation')

        # Set labels and title
        # ax.set_xlabel(f'$\epsilon_{{{component}}}$', fontsize=font, fontname='Times New Roman')
        ax.set_xlabel('Time increment', fontsize=font, fontname='Times New Roman')
        ax.set_ylabel(f'$\sigma_{{{component}}} $ (MPa)', fontsize=font, fontname='Times New Roman')

        # Calculate maximum absolute values for symmetric axis limits
        # max_strain = np.max(np.abs(strain_test[traj_ID, :, i].numpy()))*1.1
        # max_stress = np.max(np.abs(predicted_stress[traj_ID, :, i].numpy()/mag_scale))*1.1

        # Set symmetric axis limits
        # ax.set_xlim(-max_strain, max_strain)
        # ax.set_ylim(-max_stress, max_stress)

        # Add legend only to the first subplot or as needed
        if i == 2:
            ax.legend(loc='upper right', prop={'family': 'Times New Roman', 'size': font})

        # Enable grid for better readability
        ax.grid(True)
    else:
        # Hide any unused subplots (if the grid has more subplots than components)
        ax.axis('off')

plt.tight_layout()
plt.show()

print(model_y.shape)
# Create a 2x3 grid of subplots to reflect tensor symmetry
fig, axs = plt.subplots(k_inv, 1, figsize=(7, 7))

for j, ax in enumerate(axs.flat):
    if j < k_inv:
        ax.plot(model_y[:, i+j+1], 'r', label='model estimation')

        # Set labels and title
        # ax.set_xlabel(f'$\epsilon_{{{component}}}$', fontsize=font, fontname='Times New Roman')
        ax.set_xlabel('Time increment', fontsize = font, fontname='Times New Roman')
        ax.set_ylabel(f'$isv_{i+j}$', fontsize = font, fontname='Times New Roman')

        # Calculate maximum absolute values for symmetric axis limits
        # max_strain = np.max(np.abs(strain_test[traj_ID, :, i].numpy()))*1.1
        # max_stress = np.max(np.abs(predicted_stress[traj_ID, :, i].numpy()/mag_scale))*1.1

        # Set symmetric axis limits
        # ax.set_xlim(-max_strain, max_strain)
        # ax.set_ylim(-max_stress, max_stress)

        # Add legend only to the first subplot or as needed
        if i == 2:
            ax.legend(loc='upper right', prop={'family': 'Times New Roman', 'size': font})

        # Enable grid for better readability
        ax.grid(True)
    else:
        # Hide any unused subplots (if the grid has more subplots than components)
        ax.axis('off')

plt.tight_layout()
plt.show()

# plt.savefig('strain_stress_visualization.svg', dpi=600)

In [ ]:
# sample_ts = jax_time[0]
# sample_coeffs = tuple(c[200] for c in coeffs)
# interp = diffrax.CubicInterpolation(sample_ts.squeeze(-1), sample_coeffs)
# values = jax.vmap(interp.evaluate)(sample_ts)
# print(sample_ts.squeeze(-1).shape)
# # Plot comparison
# plt.figure(figsize=(6, 4))
# plt.plot(jax_time.squeeze(-1)[0], x_strain[200, :, 1], label="Original x_strain", linestyle="dashed", linewidth=6)
# plt.plot(sample_ts.squeeze(-1), values.squeeze(1)[:,2], label="Reconstructed x_strain")
# plt.xlabel("Time Step")
# plt.ylabel("Strain")
# plt.legend()
# plt.title("Comparison of Original and Reconstructed x_strain")
# plt.show()

In [ ]:
# # tester for jacobian construction
# def test_construct_jacobian():
#     # Initialize test parameters
#     stress_dim = 1  # observable states
#     k_inv = 1  # hidden states
#     width_size=64
#     num_layers=2
#     key = jr.PRNGKey(0)
    
#     # Create test instance
#     func = StateSpaceFunc(stress_dim, k_inv, width_size, num_layers, key=key)
    
#     outsize = (stress_dim+1)*stress_dim/2 + k_inv*stress_dim
#     # Create test output from network
#     # Size should be (stress_dim+1) * stress_dim/2 + k_inv * stress_dim = 6 + 6 = 12 for stress_dim=3, k_inv=2
#     test_output = jnp.arange(outsize)
    
#     # Get Jacobian
#     jacobian = func.construct_jacobian(test_output)
    
#     y = func.__call__(1, test_output, [])
#     print("Test output (network output):")
#     print(test_output)
#     print("\nConstructed Jacobian:")
#     print(jacobian)
#     print(jacobian.shape)
#     print("y:")
#     print(y)
#     print(y.shape)
    
# # Run test
# test_construct_jacobian()